In [1]:
import re
from bisect import bisect_right
import uuid
from collections import defaultdict
from datetime import datetime
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType, IntegerType, LongType, StringType, StructField, StructType, TimestampType
)
from pyspark.sql.window import Window

TIME_PARSER_POLICY = "CORRECTED"

RUN_ID = str(uuid.uuid4())
STARTED_AT = datetime.utcnow()
spark.conf.set("spark.sql.legacy.timeParserPolicy", TIME_PARSER_POLICY)


StatementMeta(, 251f226a-3497-4db3-8cec-a025105c1e25, 3, Finished, Available, Finished, False)

In [ ]:

def qident(value):
    return "`" + str(value).replace("`", "``") + "`"


def normalise(value):
    return re.sub(r"[^a-z0-9]", "", (value or "").lower())


def append_rows(table_name, rows, schema):
    if rows:
        spark.createDataFrame(rows, schema).write.format("delta").mode("append").saveAsTable(table_name)


def map_data_type(pg_type):
    value = (pg_type or "").lower().strip()
    if "[]" in value:
        return "ARRAY<STRING>"
    if any(token in value for token in ("uuid", "json", "text", "character", "varchar")):
        return "STRING"
    if value in {"smallint", "int2", "integer", "int", "int4"}:
        return "INT"
    if value in {"bigint", "int8"}:
        return "BIGINT"
    match = re.search(r"(?:numeric|decimal)\s*\((\d+)\s*,\s*(\d+)\)", value)
    if match:
        precision = min(int(match.group(1)), 38)
        scale = min(int(match.group(2)), precision)
        return f"DECIMAL({precision},{scale})"
    if "numeric" in value or "decimal" in value:
        return "DECIMAL(38,18)"
    if any(token in value for token in ("double", "float", "real")):
        return "DOUBLE"
    if "boolean" in value or value == "bool":
        return "BOOLEAN"
    if value == "date":
        return "DATE"
    if "timestamp" in value:
        return "TIMESTAMP"
    return "STRING"


def first_parsed(column, formats, parser):
    return F.coalesce(*[parser(column, fmt) for fmt in formats])


def cast_column(frame, definition):
    name = definition["column_name"]
    spark_type = map_data_type(definition["data_type"])
    if name not in frame.columns:
        return F.lit(None).cast(spark_type).alias(name)
    source = F.col(qident(name))
    if spark_type == "BOOLEAN":
        clean = F.lower(F.trim(source.cast("string")))
        return (F.when(clean.isin("true", "t", "1", "yes", "y"), F.lit(True))
            .when(clean.isin("false", "f", "0", "no", "n"), F.lit(False))
            .otherwise(F.lit(None).cast("boolean")).alias(name))
    if spark_type == "DATE":
        return first_parsed(source.cast("string"), DATE_FORMATS, F.to_date).alias(name)
    if spark_type == "TIMESTAMP":
        return first_parsed(source.cast("string"), TIMESTAMP_FORMATS, F.to_timestamp).alias(name)
    if spark_type.startswith("DECIMAL") or spark_type in {"INT", "BIGINT", "DOUBLE"}:
        return F.regexp_replace(source.cast("string"), r"[^0-9eE+\.\-]", "").cast(spark_type).alias(name)
    if spark_type == "ARRAY<STRING>":
        return F.when(source.isNull(), F.lit(None).cast("array<string>"))             .otherwise(F.split(F.regexp_replace(source.cast("string"), r"^[\{\[]|[\}\]]$", ""), r"\s*,\s*")).alias(name)
    return F.trim(source.cast("string")).alias(name)


def resolve_contract(physical_table, prefixes):
    base = physical_table.lower()
    for prefix in prefixes:
        if base.startswith(prefix):
            base = base[len(prefix):]
    matches = contracts_by_table.get(normalise(base), [])
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise ValueError(f"Ambiguous table contract for {physical_table}: {matches}")
    return None


def primary_key_columns(schema_cols):
    """Return the ordered business key defined by the schema contract."""
    return [
        column["column_name"] for column in schema_cols
        if (column.get("is_primary_key") or "").upper() == "YES"
    ]


def deduplicate_frame(frame, schema_cols):
    """Keep one row per contracted PK without triggering count jobs here."""
    key_columns = primary_key_columns(schema_cols)
    if not key_columns:
        return frame, key_columns
    if "_archive_load_ts" in frame.columns:
        ordering = F.col("_archive_load_ts").cast("timestamp").desc_nulls_last()
    elif "_ingestion_timestamp" in frame.columns:
        ordering = F.col("_ingestion_timestamp").cast("timestamp").desc_nulls_last()
    else:
        ordering = F.col(qident("export_date")).cast("timestamp").desc_nulls_last()
    window = Window.partitionBy(
        *[F.col(qident(column)) for column in key_columns]
    ).orderBy(ordering)
    ranked = frame.withColumn("_silver_row_number", F.row_number().over(window))
    return ranked.where(F.col("_silver_row_number") == 1).drop("_silver_row_number"), key_columns


def diagnostic_key_sample(frame, key_columns):
    """Collect a small identifier-only sample for operational tracing."""
    if not VERBOSE_DIAGNOSTICS or not key_columns:
        return []
    return [
        row.asDict(recursive=True)
        for row in frame.select(*[F.col(qident(column)) for column in key_columns])
            .limit(DIAGNOSTIC_KEY_SAMPLE_SIZE).collect()
    ]


def print_load_diagnostics(source_table, target_table, snapshot_date, source_date,
                           raw_frame, formatted_frame, key_columns,
                           source_count, written_count, duplicate_count,
                           formatted_export_non_null):
    """Print dates, counts and PK identifiers needed to diagnose a batch."""
    if not VERBOSE_DIAGNOSTICS:
        return
    raw_export_type = next(
        field.dataType.simpleString() for field in raw_frame.schema.fields
        if field.name == "export_date"
    )
    formatted_dates = [
        str(row["export_date"])
        for row in formatted_frame.select(F.to_date("export_date").alias("export_date"))
            .where(F.col("export_date").isNotNull()).distinct().orderBy("export_date").collect()
    ]
    print(f"  Source: {source_table}; selected export={source_date}; raw type={raw_export_type}")
    print(f"  Target: {target_table}; Gold snapshot={snapshot_date}")
    print(f"  Rows: source={source_count:,}; written={written_count:,}; duplicates removed={duplicate_count:,}")
    print(f"  Primary keys: {key_columns or ['<none>']}")
    print(f"  Primary-key sample: {diagnostic_key_sample(formatted_frame, key_columns)}")
    print(
        f"  Silver export_date: non-null={formatted_export_non_null:,}; "
        f"null={written_count - formatted_export_non_null:,}; values={formatted_dates}"
    )


def format_frame(frame, schema_cols, source_kind, source_table):
    expressions = [cast_column(frame, definition) for definition in schema_cols]
    return (frame.select(*expressions)
        .withColumn("_record_source", F.lit(source_kind))
        .withColumn("_source_table", F.lit(source_table))
        .withColumn("_silver_run_id", F.lit(RUN_ID))
        .withColumn("_silver_load_ts", F.current_timestamp()))


AUDIT_SCHEMA = StructType([
    StructField("source_kind", StringType(), False),
    StructField("source_schema", StringType(), False),
    StructField("source_table", StringType(), False),
    StructField("target_table", StringType(), True),
    StructField("export_date", TimestampType(), False),
    StructField("status", StringType(), False),
    StructField("reload", BooleanType(), False),
    StructField("attempt_count", IntegerType(), False),
    StructField("run_id", StringType(), True),
    StructField("rows_read", LongType(), True),
    StructField("rows_written", LongType(), True),
    StructField("duplicate_key_count", LongType(), True),
    StructField("started_at", TimestampType(), True),
    StructField("ended_at", TimestampType(), True),
    StructField("error_message", StringType(), True),
    StructField("last_updated_at", TimestampType(), True),
])


def audit_record(source_kind, source_schema, source_table, export_date):
    rows = (spark.table(AUDIT_TABLE)
        .where((F.col("source_kind") == source_kind)
            & (F.col("source_schema") == source_schema)
            & (F.col("source_table") == source_table)
            & (F.col("export_date") == F.lit(export_date).cast("timestamp")))
        .limit(1).collect())
    return rows[0].asDict() if rows else None


def should_skip(source_kind, source_schema, source_table, export_date):
    record = audit_record(source_kind, source_schema, source_table, export_date)
    return bool(record and record["status"] == "SUCCESS" and not record["reload"])


def audit_begin(source_kind, source_schema, source_table, target_table, export_date):
    now = datetime.utcnow()
    existing = audit_record(source_kind, source_schema, source_table, export_date)
    attempt_count = int(existing["attempt_count"] or 0) + 1 if existing else 1
    row = [(source_kind, source_schema, source_table, target_table, export_date, "RUNNING",
            bool(existing["reload"]) if existing else False, attempt_count, RUN_ID,
            None, None, None, now, None, None, now)]
    source = spark.createDataFrame(row, AUDIT_SCHEMA)
    target = DeltaTable.forName(spark, AUDIT_TABLE)
    condition = " AND ".join([
        "t.source_kind = s.source_kind", "t.source_schema = s.source_schema",
        "t.source_table = s.source_table", "t.export_date = s.export_date",
    ])
    (target.alias("t").merge(source.alias("s"), condition)
        .whenMatchedUpdate(set={
            "target_table": "s.target_table", "status": "s.status",
            "attempt_count": "s.attempt_count", "run_id": "s.run_id",
            "started_at": "s.started_at", "ended_at": "s.ended_at",
            "error_message": "s.error_message", "last_updated_at": "s.last_updated_at",
        }).whenNotMatchedInsertAll().execute())


def audit_finish(source_kind, source_schema, source_table, target_table, export_date,
                 status, rows_read=0, rows_written=0, duplicate_count=0, error_message=None):
    now = datetime.utcnow()
    existing = audit_record(source_kind, source_schema, source_table, export_date) or {}
    row = [(source_kind, source_schema, source_table, target_table, export_date, status,
            False if status == "SUCCESS" else bool(existing.get("reload", False)),
            int(existing.get("attempt_count") or 1), RUN_ID, int(rows_read), int(rows_written),
            int(duplicate_count), existing.get("started_at") or now, now,
            error_message[:4000] if error_message else None, now)]
    source = spark.createDataFrame(row, AUDIT_SCHEMA)
    target = DeltaTable.forName(spark, AUDIT_TABLE)
    condition = " AND ".join([
        "t.source_kind = s.source_kind", "t.source_schema = s.source_schema",
        "t.source_table = s.source_table", "t.export_date = s.export_date",
    ])
    (target.alias("t").merge(source.alias("s"), condition)
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

    
def logical_table_name(value):
    name = (value or "").lower()
    for prefix in TABLE_PREFIXES:
        if name.startswith(prefix):
            name = name[len(prefix):]
    return name



In [ ]:
##Silver Layer functions

def silver_table(table_name):
    return f"{SILVER_SCHEMA}.slv_{table_name.lower()}"
